In [1]:
from PIL import Image
import pytesseract
from transformers import AutoModelForCausalLM, AutoTokenizer
import datetime

from pdf2image import convert_from_path
from PIL import Image, ImageChops
import os

In [2]:
# pytesseract.pytesseract.tesseract_cmd = r'C:\Users\mosta\AppData\Local\Programs\Tesseract-OCR'

In [5]:
model_name = "Qwen/Qwen3-1.7B"

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype="auto",
    device_map="auto"
)

TypeError: expected str, bytes or os.PathLike object, not NoneType

In [ ]:
directory_name = "resume_samples_pdf"
img_directory_name = "resume_sample_imgs_concat"
pdf_files = [pdf for pdf in os.listdir(directory_name)]

In [ ]:
def image_concat(to_concat_images):

    widths, heights = zip(*(i.size for i in to_concat_images))

    max_width = max(widths)
    total_height = sum(heights)

    new_im = Image.new('RGB', (max_width, total_height))

    y_offset = 0

    for im in to_concat_images:
      new_im.paste(im, (0, y_offset))
      y_offset += im.size[1]

    return new_im
    # new_im.save('test.jpg')

def trim_whitespace(im):
    bg = Image.new(im.mode, im.size, im.getpixel((0,0)))
    diff = ImageChops.difference(im, bg)
    diff = ImageChops.add(diff, diff, 2.0, -100)
    bbox = diff.getbbox()
    if bbox:
        return im.crop(bbox)
    return im

image_list = []
for pdf in pdf_files:
    file_path = os.path.join(directory_name, pdf)

    images = convert_from_path(file_path, dpi=150, fmt="png")
    images = [trim_whitespace(img) for img in images]
    final_image = image_concat(images)

    final_image.save(f"{img_directory_name}/{pdf.rsplit('.', 1)[0]}.png", "PNG")
    image_list.append(images)

In [ ]:
# Use a raw string with double-braces for the JSON structure
def prompter(ocr_output):
    return f"""
    Extract and structure the information from the following OCR text into a clean JSON format.

    ### OCR TEXT START ###
    {ocr_output}
    ### OCR TEXT END ###

    ### INSTRUCTIONS ###
    1. Clean up OCR errors and normalize text.
    2. For 'Work Experience' and 'Education', concatenate all descriptions/details into a single coherent paragraph for each entry.
    3. If information is missing, use an empty string "" or an empty list [].
    4. Output ONLY valid JSON. Do not include introductory text.

    ### TARGET JSON STRUCTURE ###
    {{
        "personal_info": {{
            "first_name": "",
            "last_name": "",
            "email": "",
            "phone": "",
            "address": "",
            "city": "",
            "country": ""
        }},
        "experiences": [
            {{
                "job_title": "",
                "company": "",
                "dates": "",
                "responsibilities": ""
            }}
        ],
        "education": [
            {{
                "degree": "",
                "institution": "",
                "graduation_date": "",
                "details": ""
            }}
        ],
        "skills": {{
            "technical": "",
            "soft": "",
            "interests": ""
        }}
    }}
    """

In [ ]:
for f in os.listdir(f'{img_directory_name}'):

    time_start = datetime.datetime.now()
    print(f"Processing {f}")
    tesseract_output = pytesseract.image_to_string(f'{img_directory_name}/{f}')
    time_end = datetime.datetime.now()
    time_difference = (time_end - time_start)

    minutes = divmod(time_difference.total_seconds(), 60)
    print(f"OCR Model Execution time is {minutes[0]} minutes and {minutes[1]} seconds.")
    messages = [
        {"role": "user", "content": prompter(tesseract_output)}
    ]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False # Switches between thinking and non-thinking modes. Default is True.
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)

    # conduct text completion
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=1536
    )

    time_end = datetime.datetime.now()
    time_difference = (time_end - time_start)

    minutes = divmod(time_difference.total_seconds(), 60)
    print(f"LLM Model Execution time is {minutes[0]} minutes and {minutes[1]} seconds.")

    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist()

    # parsing thinking content
    try:
        # rindex finding 151668 (</think>)
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0

    # thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    content = tokenizer.decode(output_ids[index:], skip_special_tokens=True).strip("\n")

    # print("thinking content:", thinking_content)
    # print("content:", content)

    # print(json_output)
    with open(f"sample_output_tesseract/{f.rsplit('.', 1)[0]}.json", "w", encoding='utf-8') as file_write:
        file_write.write(content)
    print(f"Saved to sample_output_tesseract/{f.rsplit('.', 1)[0]}.json")